# Phase 3 — Model Training 

**What this notebook does:**
- Trains a baseline XGBoost model using SageMaker Training Jobs
- Tracks experiments with SageMaker Experiments

## 1. Load config & Session

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris
import json

with open("../data/config.json") as f:
    cfg = json.load(f)
 
ROLE = cfg["ROLE"]
BUCKET = cfg["BUCKET"]
PREFIX = cfg["PREFIX"]
REGION = cfg["REGION"]
SCALE_POS_WEIGHT  = cfg["SCALE_POS_WEIGHT"]

boto_session = boto3.Session(region_name=REGION)
sagemaker_session = Session(boto_session=boto_session)


TRAIN_S3 = f"s3://{BUCKET}/{PREFIX}/processed/train/"
VAL_S3  = f"s3://{BUCKET}/{PREFIX}/processed/test/"

print(f"Train data : {TRAIN_S3}")
print(f"Test data  : {VAL_S3}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

[05/29/26 18:34:01] INFO     Found credentials in shared    credentials.py:1392
                             credentials file:                                 
                             ~/.aws/credentials                                
Train data : s3://credit-risk-mlops-svp/credit-risk/processed/train/
Test data  : s3://credit-risk-mlops-svp/credit-risk/processed/test/
scale_pos_weight: 11.4


## 2. Baseline training with SageMaker Experiments

In [21]:
from sagemaker.core import image_uris

xgb_image = image_uris.retrieve(
    framework="xgboost",
    region=REGION,
    version="1.7-1"
)
print(f"XGBoost Image URI: {xgb_image}")

[05/29/26 18:30:13] INFO     Ignoring unnecessary instance    image_uris.py:535
                             type: None.                                       
XGBoost Image URI: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [37]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData, OutputDataConfig    
import sagemaker.train.model_trainer as mt_module
import os

# Store original method
_original_prepare = mt_module.ModelTrainer._prepare_train_script

def _patched_prepare_train_script(self, tmp_dir, source_code, distributed=None):
    """Patched version that forces LF line endings on Windows."""
    _original_prepare(self, tmp_dir, source_code, distributed)
    
    # Fix the generated sm_train.sh line endings
    train_script_path = os.path.join(tmp_dir.name, "sm_train.sh")
    if os.path.exists(train_script_path):
        with open(train_script_path, "r") as f:
            content = f.read()
        with open(train_script_path, "w", newline="\n") as f:  # newline="\n" forces LF
            f.write(content)

# Apply the patch
mt_module.ModelTrainer._prepare_train_script = _patched_prepare_train_script

print("Patch applied — sm_train.sh will be written with LF line endings")

source_code = SourceCode(
    source_dir="../src",
    command ="python train.py",
    requirements="requirements.txt"
)

compute = Compute(
    instance_type = "ml.m5.xlarge",
    instance_count = 1
)

output_config = OutputDataConfig(
    s3_output_path = f"s3://{BUCKET}/{PREFIX}/models/"
)

model_trainer = ModelTrainer(
    training_image=xgb_image,
    role = ROLE,
    sagemaker_session = sagemaker_session,
    source_code = source_code,
    compute = compute,
    output_data_config = output_config,
    base_job_name= "credit-training",
    hyperparameters={
        "max_depth":             6,
        "eta":                   0.1,
        "min_child_weight":      5,
        "subsample":             0.8,
        "colsample_bytree":      0.8,
        "num_round":             300,
        "early_stopping_rounds": 20,
        "scale_pos_weight":      SCALE_POS_WEIGHT
    }
)

train_data = InputData(
    channel_name="train",
    data_source= TRAIN_S3
)

val_data = InputData(
    channel_name= "validation",
    data_source = VAL_S3
)

print("Starting baseline training job...")

training_job = model_trainer.train(
    input_data_config= [train_data, val_data],
    wait = True,
    logs = True
)

baseline_job_name = model_trainer._latest_training_job.training_job_name
print(f"\nBaseline complete: {baseline_job_name}")

Patch applied — sm_train.sh will be written with LF line endings
[05/30/26 12:01:59] INFO     StoppingCondition not provided.    defaults.py:128
                             Using default:                                    
                             max_runtime_in_seconds=3600                       
                             max_wait_time_in_seconds=None                     
                             max_pending_time_in_seconds=None                  
                    INFO     OutputDataConfig compression type  defaults.py:165
                             not provided. Using default:                      
                             GZIP                                              
                    INFO     Training image URI:           model_trainer.py:553
                             683313688378.dkr.ecr.us-east-                     
                             1.amazonaws.com/sagemaker-xgb                     
                             oost:1.7-1                

In [ ]:
# Checking the status of the training job
import boto3
import json

with open("../data/config.json") as f:
    cfg = json.load(f)

sm_client = boto3.client("sagemaker", region_name=cfg["REGION"])

# Get the failed job name
job_name = model_trainer._latest_training_job.training_job_name

desc = sm_client.describe_training_job(TrainingJobName=job_name)

print(f"Status         : {desc['TrainingJobStatus']}")
print(f"Failure Reason : {desc.get('FailureReason', 'N/A')}")
print(f"\nSecondary status transitions:")
for t in desc.get("SecondaryStatusTransitions", []):
    print(f"  {t['Status']:<30} {t.get('StatusMessage', '')}")

Status         : Completed
Failure Reason : N/A

Secondary status transitions:
  Starting                       Preparing the instances for training
  Training                       Training image download completed. Training in progress.
  Uploading                      Uploading generated training model
  Completed                      Training job completed


In [39]:
# Extract baseline AUC from training job metrics
sm_client = boto_session.client("sagemaker")

job_desc  = sm_client.describe_training_job(TrainingJobName=baseline_job_name)
metrics   = {m["MetricName"]: m["Value"] for m in job_desc.get("FinalMetricDataList", [])}

print("Baseline model metrics:")
for k, v in metrics.items():
    print(f"  {k:<35} {v:.4f}")

baseline_auc = metrics.get("validation:auc", 0)
print(f"\nBaseline AUC-ROC: {baseline_auc:.4f}")
print("Target: > 0.75 to proceed to tuning")

Baseline model metrics:
  validation:auc                      0.7636
  train:auc                           0.8311
  validation:aucpr                    0.2530
  train:aucpr                         0.3261

Baseline AUC-ROC: 0.7636
Target: > 0.75 to proceed to tuning


In [5]:
# Save to config 
import json

with open("../data/config.json") as f:
    cfg = json.load(f)

cfg["BASELINE_JOB_NAME"] = baseline_job_name
cfg["BASELINE_AUC"] = baseline_job_name

with open("../data/config.json", "w") as f:
    json.dump(cfg, f, indent=2)

